In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, Optional, Tuple, Union

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from matplotlib.patches import Circle, FancyArrow, Rectangle


def plot_dynamic_rank_maps_filtered_by_capacity(
    *,
    # --- Dynamic inputs (tract × time) ---
    availability_csv: Union[str, Path],
    usage_csv: Union[str, Path],
    idle_time_csv: Union[str, Path],

    # --- Capacity input (defines tract universe like your notebook) ---
    capacity_csv: Union[str, Path],

    # --- Geometry ---
    ny_tract_shp: Union[str, Path],

    # --- Output ---
    output_dir: Union[str, Path],

    # --- Column names ---
    csv_tract_col: str = "census_tract",
    shp_geoid_col: str = "GEOID",
    time_col: str = "time_slot",

    # Dynamic metric columns (edit to match your CSV column names)
    availability_value_col: str = "total_vehicle_available_norm",
    usage_value_col: str = "num_trips_starting_norm",
    idle_value_col: str = "avg_idle_time_norm",

    # How to evaluate the “dynamic value” per tract (mean across week by default)
    dynamic_agg: str = "mean",  # "mean", "median", "sum", "max", "min"

    # NYC borough prefixes
    nyc_borough_prefixes: Tuple[str, ...] = ("36061", "36047", "36081", "36005"),

    # Capacity-based tract filter (same idea as notebook)
    station_count_col: str = "num_station",
    min_stations: int = 1,
    gate_col: str = "total_capacity_norm",
    drop_zeros: bool = True,

    # Plot styling
    figsize: Tuple[int, int] = (10, 10),
    cmap: str = "RdYlBu",
    edgecolor: str = "black",
    linewidth: float = 0.30,
    alpha: float = 0.90,
    legend_loc: str = "upper left",
    add_basemap: bool = True,

    # Overlays
    add_compass: bool = True,
    add_scalebar: bool = True,
    scalebar_segments_km: Tuple[int, int, int] = (0, 2, 5),
) -> Dict[str, Path]:
    """
    Creates 3 dynamic rank-decile tract maps:
      (a) Dynamic Vehicle Availability Rank
      (b) Dynamic Usage (Trips Starting) Rank
      (c) Dynamic Idle Time Rank

    Dynamic value evaluation:
      value_per_tract = aggregate(metric over all time slots)  <-- controlled by dynamic_agg

    Filtering (same as your notebook):
      1) Filter shapefile to NYC borough GEOID prefixes
      2) Build tract universe from capacity_csv where num_station >= min_stations
      3) Merge and apply gate_col (and optionally drop zeros)
      4) Only those tracts are mapped
    """

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # ----------------------------
    # Helpers
    # ----------------------------
    def to_geoid11(s: pd.Series) -> pd.Series:
        return (
            s.astype(str)
            .str.strip()
            .str.replace(r"\.0$", "", regex=True)
            .str.replace(r"\s+", "", regex=True)
            .str.zfill(11)
        )

    def add_decile_labels(gdf: gpd.GeoDataFrame, value_col: str, label_col: str) -> None:
        p = gdf[value_col].rank(pct=True, method="average")
        bins = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
        labels = [
            "1-10%", "11-20%", "21-30%", "31-40%", "41-50%",
            "51-60%", "61-70%", "71-80%", "81-90%", "91-100%"
        ]
        gdf[label_col] = pd.cut(p, bins=bins, labels=labels, include_lowest=True)

    def add_compass_rose(ax, xlim, ylim, frac_pos=(0.88, 0.88), size_frac=0.075):
        width = xlim[1] - xlim[0]
        height = ylim[1] - ylim[0]
        cx = xlim[0] + frac_pos[0] * width
        cy = ylim[0] + frac_pos[1] * height
        R = size_frac * min(width, height)

        outer = Circle((cx, cy), R, facecolor="white", edgecolor="black", linewidth=1.2, zorder=10)
        inner = Circle((cx, cy), R * 0.88, facecolor="white", edgecolor="black", linewidth=0.8, zorder=11)
        ax.add_patch(outer)
        ax.add_patch(inner)

        ax.text(cx, cy + R + 0.012 * height, "N", ha="center", va="bottom",
                fontsize=12, fontweight="bold", zorder=12)
        ax.text(cx, cy - R - 0.012 * height, "S", ha="center", va="top", fontsize=10, zorder=12)
        ax.text(cx - R - 0.006 * width, cy, "W", ha="right", va="center", fontsize=10, zorder=12)
        ax.text(cx + R + 0.006 * width, cy, "E", ha="left", va="center", fontsize=10, zorder=12)

        ax.add_patch(
            FancyArrow(
                cx, cy - 0.18 * R, 0, 0.72 * R,
                width=0.0,
                head_width=0.22 * R,
                head_length=0.28 * R,
                length_includes_head=True,
                facecolor="black",
                edgecolor="black",
                zorder=13,
            )
        )

    def add_scalebar_km(ax, xlim, ylim, frac_pos=(0.72, 0.06), bar_height_frac=0.012, segments_km=(0, 2, 5)):
        # EPSG:3857 => meters
        width = xlim[1] - xlim[0]
        height = ylim[1] - ylim[0]
        x0 = xlim[0] + frac_pos[0] * width
        y0 = ylim[0] + frac_pos[1] * height

        total_m = segments_km[-1] * 1000.0
        bar_h = bar_height_frac * height

        seg1_m = (segments_km[1] - segments_km[0]) * 1000.0
        seg2_m = (segments_km[2] - segments_km[1]) * 1000.0

        r1 = Rectangle((x0, y0), seg1_m, bar_h, facecolor="black", edgecolor="black", linewidth=0.8, zorder=10)
        r2 = Rectangle((x0 + seg1_m, y0), seg2_m, bar_h, facecolor="white", edgecolor="black", linewidth=0.8, zorder=10)
        ax.add_patch(r1)
        ax.add_patch(r2)

        ax.plot([x0, x0], [y0, y0 + bar_h], color="black", linewidth=1.0, zorder=11)
        ax.plot([x0 + seg1_m, x0 + seg1_m], [y0, y0 + bar_h], color="black", linewidth=1.0, zorder=11)
        ax.plot([x0 + total_m, x0 + total_m], [y0, y0 + bar_h], color="black", linewidth=1.0, zorder=11)

        ax.text(x0, y0 - 0.012 * height, f"{segments_km[0]}", ha="center", va="top", fontsize=10, zorder=12)
        ax.text(x0 + seg1_m, y0 - 0.012 * height, f"{segments_km[1]} km", ha="center", va="top", fontsize=10, zorder=12)
        ax.text(x0 + total_m, y0 - 0.012 * height, f"{segments_km[2]} km", ha="center", va="top", fontsize=10, zorder=12)
        ax.text(x0 + total_m / 2, y0 + bar_h + 0.008 * height, "Scale", ha="center", va="bottom", fontsize=10, zorder=12)

    def aggregate_dynamic(df: pd.DataFrame, value_col: str) -> pd.DataFrame:
        if dynamic_agg not in {"mean", "median", "sum", "max", "min"}:
            raise ValueError("dynamic_agg must be one of: mean, median, sum, max, min")

        if csv_tract_col not in df.columns:
            raise KeyError(f"Missing tract column '{csv_tract_col}' in dynamic CSV.")
        if value_col not in df.columns:
            raise KeyError(f"Missing value column '{value_col}' in dynamic CSV.")
        if time_col in df.columns:
            # time is not required for aggregation, but we validate parse if present
            df[time_col] = pd.to_datetime(df[time_col], errors="coerce")

        df[csv_tract_col] = to_geoid11(df[csv_tract_col])

        # one value per tract
        grouped = df.groupby(csv_tract_col)[value_col]
        if dynamic_agg == "mean":
            out = grouped.mean()
        elif dynamic_agg == "median":
            out = grouped.median()
        elif dynamic_agg == "sum":
            out = grouped.sum()
        elif dynamic_agg == "max":
            out = grouped.max()
        else:  # min
            out = grouped.min()

        return out.reset_index().rename(columns={value_col: f"{value_col}__{dynamic_agg}"})

    def load_capacity_universe() -> np.ndarray:
        cap_raw = pd.read_csv(capacity_csv)
        if csv_tract_col not in cap_raw.columns:
            raise KeyError(f"capacity_csv missing '{csv_tract_col}'")
        if station_count_col not in cap_raw.columns:
            raise KeyError(f"capacity_csv missing '{station_count_col}'")
        if gate_col not in cap_raw.columns:
            raise KeyError(f"capacity_csv missing '{gate_col}'")

        cap_raw[csv_tract_col] = to_geoid11(cap_raw[csv_tract_col])

        # aggregate per tract (same spirit as notebook)
        agg = {station_count_col: "sum", gate_col: "mean"}
        cap = cap_raw.groupby(csv_tract_col, as_index=False).agg(agg)

        tracts_with_stations = cap.loc[cap[station_count_col].fillna(0) >= min_stations, csv_tract_col].unique()
        if len(tracts_with_stations) == 0:
            raise ValueError(f"0 tracts found with {station_count_col} >= {min_stations}")
        return tracts_with_stations, cap

    def prep_base_gdf(cap_universe, cap_agg) -> gpd.GeoDataFrame:
        tracts = gpd.read_file(ny_tract_shp)
        if shp_geoid_col not in tracts.columns:
            raise KeyError(f"Tract shapefile missing '{shp_geoid_col}'")

        tracts[shp_geoid_col] = to_geoid11(tracts[shp_geoid_col])
        tracts_nyc = tracts[tracts[shp_geoid_col].str.startswith(nyc_borough_prefixes)].copy()
        tracts_nyc = tracts_nyc[tracts_nyc[shp_geoid_col].isin(cap_universe)].copy()
        if tracts_nyc.empty:
            raise ValueError("After NYC + capacity-universe filtering, 0 tracts remain. Check GEOID matching.")

        gdf = tracts_nyc.merge(cap_agg, left_on=shp_geoid_col, right_on=csv_tract_col, how="left")

        # gate
        gdf = gdf.dropna(subset=[gate_col]).copy()
        if drop_zeros:
            gdf = gdf[gdf[gate_col] > 0].copy()

        if gdf.empty:
            raise ValueError("0 tracts remain after gate filtering. Try drop_zeros=False to debug.")

        return gdf

    def plot_rank_map(gdf_web: gpd.GeoDataFrame, value_col: str, title: str, filename: str) -> Path:
        tmp = gdf_web.dropna(subset=[value_col]).copy()
        if tmp.empty:
            raise ValueError(f"No non-null values to plot for '{value_col}'")

        decile_col = f"{value_col}__decile"
        add_decile_labels(tmp, value_col, decile_col)

        xmin, ymin, xmax, ymax = tmp.total_bounds
        xlim, ylim = (xmin, xmax), (ymin, ymax)

        fig, ax = plt.subplots(figsize=figsize)
        tmp.plot(
            ax=ax,
            column=decile_col,
            categorical=True,
            legend=True,
            cmap=cmap,
            edgecolor=edgecolor,
            linewidth=linewidth,
            alpha=alpha,
            legend_kwds={"loc": legend_loc},
        )

        ax.set_xlim(xlim)
        ax.set_ylim(ylim)

        if add_basemap:
            ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik)
            ax.set_xlim(xlim)
            ax.set_ylim(ylim)

        if add_compass:
            add_compass_rose(ax, xlim, ylim)

        if add_scalebar:
            add_scalebar_km(ax, xlim, ylim, segments_km=scalebar_segments_km)

        ax.set_title(title, fontsize=15, fontweight="bold")
        ax.axis("off")

        outpath = output_dir / filename
        plt.savefig(outpath, dpi=300, bbox_inches="tight")
        plt.close(fig)
        return outpath

    # ----------------------------
    # Build capacity tract universe (like notebook)
    # ----------------------------
    cap_universe, cap_agg = load_capacity_universe()
    base_gdf = prep_base_gdf(cap_universe, cap_agg)

    # ----------------------------
    # Aggregate dynamic values (define evaluation here)
    # ----------------------------
    avail_df = pd.read_csv(availability_csv)
    usage_df = pd.read_csv(usage_csv)
    idle_df = pd.read_csv(idle_time_csv)

    avail_agg = aggregate_dynamic(avail_df, availability_value_col)
    usage_agg = aggregate_dynamic(usage_df, usage_value_col)
    idle_agg = aggregate_dynamic(idle_df, idle_value_col)

    # merge onto geometry
    gdf = base_gdf.merge(avail_agg, on=csv_tract_col, how="left")
    gdf = gdf.merge(usage_agg, on=csv_tract_col, how="left")
    gdf = gdf.merge(idle_agg, on=csv_tract_col, how="left")

    # project once for mapping
    gdf_web = gdf.to_crs(epsg=3857)

    # columns after aggregation
    avail_col = f"{availability_value_col}__{dynamic_agg}"
    usage_col = f"{usage_value_col}__{dynamic_agg}"
    idle_col = f"{idle_value_col}__{dynamic_agg}"

    saved: Dict[str, Path] = {}
    saved["availability"] = plot_rank_map(
        gdf_web, avail_col,
        title="Dynamic Vehicle Availability Rank",
        filename=f"DYNAMIC_AVAILABILITY_RANK__{dynamic_agg}.png",
    )
    saved["usage_starting"] = plot_rank_map(
        gdf_web, usage_col,
        title="Dynamic Usage (Trips Starting) Rank",
        filename=f"DYNAMIC_USAGE_STARTING_RANK__{dynamic_agg}.png",
    )
    saved["idle_time"] = plot_rank_map(
        gdf_web, idle_col,
        title="Dynamic Idle Time Rank",
        filename=f"DYNAMIC_IDLE_TIME_RANK__{dynamic_agg}.png",
    )

    return saved


In [3]:
saved = plot_dynamic_rank_maps_filtered_by_capacity(
    availability_csv=r"D:\Research Fellowship\NYC_Docked_Output_v2\availability__norm__tract.csv",
    usage_csv=r"D:\Research Fellowship\NYC_Docked_Output_v2\usage_norm_hourly_tract.csv",
    idle_time_csv=r"D:\Research Fellowship\NYC_Docked_Output_v2\idle_time_norm_tract.csv",
    capacity_csv=r"D:\Research Fellowship\NYC_Docked_Output_v2\capacity_tract_norm.csv",
    ny_tract_shp=r"D:\Research Fellowship\Summer Research Stuff\Clean_Utilities\Capacity\NYC\tl_2024_36_tract.shp",
    output_dir="DYNAMIC_TRACT_MAPS_OUT",
    dynamic_agg="mean",

    # ✅ IMPORTANT FIX
    usage_value_col="trips_starting_norm",
)
print(saved)


{'availability': WindowsPath('DYNAMIC_TRACT_MAPS_OUT/DYNAMIC_AVAILABILITY_RANK__mean.png'), 'usage_starting': WindowsPath('DYNAMIC_TRACT_MAPS_OUT/DYNAMIC_USAGE_STARTING_RANK__mean.png'), 'idle_time': WindowsPath('DYNAMIC_TRACT_MAPS_OUT/DYNAMIC_IDLE_TIME_RANK__mean.png')}
